# 01, U-Net void segmentation
Author: Karan Homayounfar, UWE Bristol, Nexus team, NCC AI Hackathon

U-Net is the main model. Ronneberger et al. (2015) showed that an encoder-decoder with skip connections preserves fine spatial detail lost by pooling alone, which matters here since voids are small and thin relative to the 256x256 frame. A resnet encoder pretrained on ImageNet is used rather than training the encoder from scratch, since 4,000 labelled images is small for a segmentation task and transfer learning reduces the risk of overfitting a from-scratch encoder in the time available.

Compared to a from-scratch CNN, the pretrained-encoder U-Net converges faster and needs less data to reach a usable Dice score, which is why published CFRP defect work (Ronneberger's descendants: 3D U-Net on micro-CT voids, Attention U-Net on thermographic CFRP defects) uses the same family of architecture rather than something exotic.

Scoring gate: Dice_void >= 0.8 is a gate, not a target, per NCC's own materials, extra Dice past 0.8 does not improve the final score. Severity/pass-fail accuracy (notebook 02) is the harder, score-determining part.

In [ ]:
!pip install -q segmentation-models-pytorch
# needs Internet turned on in notebook Settings (right sidebar), off by default on Kaggle
# first-time pip install on Kaggle may need phone verification on your account

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

DATA_DIR    = '/kaggle/input/datasets/karanhomayounfar1/ncc-composites-defect/Data sets'
OUT_DIR     = '/kaggle/working'
FIGURES_DIR = '/kaggle/working/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load data

In [ ]:
import os as _os
# sanity check before running collect_samples: confirm the real mounted path,
# Kaggle sometimes adds or drops a folder level depending on how the zip was made
print(_os.path.exists(DATA_DIR), DATA_DIR)
if _os.path.exists('/kaggle/input/datasets/karanhomayounfar1/ncc-composites-defect'):
    print(_os.listdir('/kaggle/input/datasets/karanhomayounfar1/ncc-composites-defect'))

In [ ]:
TRAIN_SETS = ['Data set I', 'Data set II', 'Data set III']
SUBSETS = ['Original data set', 'Augmented data set']
VOID = 2
NUM_CLASSES = 3

def collect_samples():
    samples = []
    for dset in TRAIN_SETS:
        for subset in SUBSETS:
            base = Path(DATA_DIR) / dset / subset
            meta_path = base / 'metadata.csv'
            if not meta_path.exists():
                continue
            meta = pd.read_csv(meta_path)
            for _, row in meta.iterrows():
                img_name = row['image_id']
                mask_name = Path(img_name).stem + '.png'
                img_path = base / 'Images' / img_name
                mask_path = base / 'Masks' / mask_name
                if img_path.exists() and mask_path.exists():
                    samples.append({'image': img_path, 'mask': mask_path,
                                     'um_per_px': float(row['um_per_pixel'])})
    return samples

samples = collect_samples()
print(f'Collected {len(samples)} labelled image/mask pairs')

rng = np.random.default_rng(42)
idx = rng.permutation(len(samples))
n_val = int(len(samples) * 0.15)
val_samples = [samples[i] for i in idx[:n_val]]
train_samples = [samples[i] for i in idx[n_val:]]
print(f'Train: {len(train_samples)}  Val: {len(val_samples)}')

In [ ]:
class VoidSegDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = np.array(Image.open(s['image']).convert('L'), dtype=np.float32) / 255.0
        mask = np.array(Image.open(s['mask']))
        if mask.ndim == 3:
            mask = mask[..., 0]
        return (torch.from_numpy(img).unsqueeze(0),
                torch.from_numpy(mask.astype(np.int64)),
                s['um_per_px'], str(s['image'].name))

train_loader = DataLoader(VoidSegDataset(train_samples), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(VoidSegDataset(val_samples), batch_size=16, shuffle=False, num_workers=2)

img, mask, um, name = VoidSegDataset(train_samples)[0]
print(f'Sample: {name}  image={tuple(img.shape)}  mask={tuple(mask.shape)}  '
      f'classes={sorted(mask.unique().tolist())}  um_per_px={um}')

## 2. Train

In [ ]:
def dice_void_score(pred_logits, mask):
    pred = pred_logits.argmax(dim=1)
    p, g = (pred == VOID), (mask == VOID)
    inter = (p & g).sum().item()
    denom = p.sum().item() + g.sum().item()
    return 1.0 if denom == 0 else 2.0 * inter / denom

def focal_dice_loss(logits, target, class_weights, gamma=2.0):
    ce = torch.nn.functional.cross_entropy(logits, target, weight=class_weights, reduction='none')
    pt = torch.exp(-ce)
    focal = ((1 - pt) ** gamma * ce).mean()
    probs = torch.softmax(logits, dim=1)[:, VOID]
    void_target = (target == VOID).float()
    inter = (probs * void_target).sum()
    dice = 1 - (2 * inter + 1) / (probs.sum() + void_target.sum() + 1)
    return focal + dice

model = smp.Unet(encoder_name='resnet18', encoder_weights='imagenet',
                  in_channels=1, classes=NUM_CLASSES).to(DEVICE)

# matrix and fibre dominate pixel counts, void is rare and matters most
class_weights = torch.tensor([1.0, 1.0, 5.0], device=DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20
history = {'train_loss': [], 'val_dice': []}
best_dice = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for img, mask, _, _ in train_loader:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        opt.zero_grad()
        logits = model(img)
        loss = focal_dice_loss(logits, mask, class_weights)
        loss.backward()
        opt.step()
        total_loss += loss.item()

    model.eval()
    dices = []
    with torch.no_grad():
        for img, mask, _, _ in val_loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            dices.append(dice_void_score(model(img), mask))

    mean_dice = float(np.mean(dices)) if dices else 0.0
    mean_loss = total_loss / len(train_loader)
    history['train_loss'].append(mean_loss)
    history['val_dice'].append(mean_dice)
    print(f'Epoch {epoch+1}/{EPOCHS}  train_loss={mean_loss:.4f}  val_dice_void={mean_dice:.4f}')

    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save(model.state_dict(), f'{OUT_DIR}/best_model.pth')
        print(f'  saved best_model.pth (Dice {best_dice:.4f})')

print(f'\nBest val Dice_void: {best_dice:.4f}  (gate is 0.8)')

## 3. Training curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['train_loss'], color='#6a9e4f')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train loss (focal + Dice)')
axes[0].set_title('Training loss')

axes[1].plot(history['val_dice'], color='#6a9e4f')
axes[1].axhline(0.8, color='k', linestyle='--', linewidth=0.8, label='gate (0.8)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val Dice_void')
axes[1].set_title('Validation Dice, void class')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/training_curve.png', bbox_inches='tight')
plt.show()

## 4. Example prediction

In [ ]:
model.load_state_dict(torch.load(f'{OUT_DIR}/best_model.pth', map_location=DEVICE))
model.eval()

img, mask, um, name = VoidSegDataset(val_samples)[0]
with torch.no_grad():
    pred = model(img.unsqueeze(0).to(DEVICE)).argmax(dim=1).cpu().squeeze(0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img.squeeze(0), cmap='gray')
axes[0].set_title('Input micrograph')
axes[1].imshow(mask, cmap='viridis', vmin=0, vmax=2)
axes[1].set_title('Ground truth (0=matrix,1=fibre,2=void)')
axes[2].imshow(pred, cmap='viridis', vmin=0, vmax=2)
axes[2].set_title('Prediction')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/example_prediction.png', bbox_inches='tight')
plt.show()

## 5. Save metrics

In [ ]:
import pickle

unet_metrics = {
    'model': 'U-Net (resnet18 encoder, ImageNet pretrained)',
    'epochs': EPOCHS,
    'train_samples': len(train_samples),
    'val_samples': len(val_samples),
    'best_val_dice_void': best_dice,
    'dice_gate': 0.8,
    'gate_cleared': best_dice >= 0.8,
}

with open(f'{OUT_DIR}/unet_metrics.pkl', 'wb') as f:
    pickle.dump(unet_metrics, f)

print('Saved best_model.pth and unet_metrics.pkl to', OUT_DIR)
print('\n=== U-NET SUMMARY ===')
for k, v in unet_metrics.items():
    print(f'  {k}: {v}')
print('\nNext: notebook 02 runs severity.py (both NCC formulas) on these predictions.')